In [1]:
import fitz, os, re, sys, torch
import pandas as pd

sys.path.append(os.path.join(os.getcwd(), 'src'))
import config
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
from huggingface_hub import login

from dotenv import load_dotenv

load_dotenv()

login(os.environ['HF_TOKEN'])

def get_nlp_tools() -> tuple[RecursiveCharacterTextSplitter, SentenceTransformer]:
    #loading embeddings model
    # Check if CUDA is available
    device = "cuda" if torch.cuda.is_available() else "cpu"

    embeddings = SentenceTransformer("google/embeddinggemma-300m", device=device)
    #Creating text splitter
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=int(config.db_config['chunk_size']),
        chunk_overlap=int(config.db_config['chunk_overlap'])
    )
    return splitter, embeddings

splitter, embeddings = get_nlp_tools()

c:\Users\User\miniconda3\envs\motos\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
Loading weights: 100%|██████████| 314/314 [00:00<00:00, 22928.07it/s]


In [2]:
def get_output_path(base_path: str, file: str) -> str:
    start = base_path.split('\\input\\')[0]
    end = base_path.split('\\input\\')[-1]
    output_path = os.path.join(start, 'output', end)
    return os.path.join(output_path, f"{file.split('.')[0]}.parquet")

def process_pdf(text: list[str], base_path: str, file: str) -> None:
    text = [re.sub('(\n* *\n)+', '\n', page) for page in text]
    text = [re.sub(' +', ' ', page).strip().lower() for page in text]
    text = [[re.sub('\n', ' ', chunk) for chunk in splitter.split_text(page)] for page in text]
    text = [pd.DataFrame({
            'text': page,
            'page': [i+1 for _ in range(len(page))],
            'chunk': range(1, len(page)+1)
        }) for i, page in enumerate(text) if len(page)]
    text = pd.concat(text).reset_index(drop=True).assign(file=file)
    text["embedding"] = [emb.tolist() for emb in embeddings.encode(text.text.tolist())]
    output_path = get_output_path(base_path, file)
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    text.to_parquet(output_path)

def process_database(base_path: str, errors: dict):
    folder = os.path.dirname(base_path)
    file = os.path.basename(base_path)
    output_path = get_output_path(folder, file)
    if os.path.isdir(base_path):
        for element in os.listdir(base_path):
            process_database(os.path.join(base_path, element), errors)
    elif not os.path.exists(output_path):
        try:
            print(f'processing file {file}')
            doc = fitz.open(os.path.join(base_path, file))
            text = [page.get_text() for page in doc]
            process_pdf(folder, file, text)
        except Exception as e:
            errors[base_path] = str(e)

In [3]:
base_path, errors = r'C:\Users\User\Desktop\projects\Chatbot-Taller-Motos\input', dict()
process_database(base_path, errors)

processing file BAJAJ PLATINA 100 MANUAL TALLER 2.pdf
processing file BAJAJ PULSAR 180 MANUAL TALLER.pdf
processing file BAJAJ PULSAR NS200 - AS200 MANUAL USUARIO.pdf
processing file BMW 650 DIAGRAMA ELECTRICO.jpg
processing file BMW F800GS WIRING DIAGRAM 2010.jpg
processing file BMW F800GS WIRING DIAGRAM.JPG
processing file HARLEY DAVIDSON SPORTER (59-85 V-TWINS 86-87) GLIDES &ELECTRA GLIDES (59-84) EVOLUTION V-TWINS (84-86) MANUAL TALLER.pdf
processing file HERO GLAMOUR MANUAL TALLER.pdf
processing file HERO HF DELUXE MANUAL TALLER.pdf
processing file HERO HONDA CD100 MANUAL TALLER.pdf
processing file HERO HUNK 150 MANUAL TALLER.pdf
processing file HERO HUNK160R FI MANUAL TALLER 2022.pdf
processing file HERO KARIZMA ZMR 225 MANUAL TALLER.pdf
processing file HERO PASION PRO MANUAL TALLER.pdf
processing file HONDA C70 MANUAL TALLER 1982.pdf
processing file HONDA CB750F2 MANUAL TALLER 1992.pdf
processing file HONDA CBR250 MANUAL TALLER.pdf
processing file HONDA CBR600 (K00) MANUAL TALLE

In [4]:
import pytesseract
import cv2
import numpy as np

pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

def process_image(image) -> np.array:
    # converting bytes → NumPy array
    image = np.frombuffer(image, np.uint8)

    # encoding the image with OpenCV
    image = cv2.imdecode(image, cv2.IMREAD_COLOR)

    image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    return cv2.adaptiveThreshold(
        image, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY, 31, 2
    )

def get_pdf_images(file: str) -> list:
    doc = fitz.open(file)

    pdf_images = []
    # Iterar por las páginas
    for page_index in range(len(doc)):
        page = doc[page_index]
        # Obtener las imágenes de la página
        images = page.get_images(full=True)
        
        for img in images:
            xref = img[0]  # referencia interna de la imagen
            base_image = doc.extract_image(xref)
            pdf_images.append(process_image(base_image["image"]))
    return pdf_images

In [5]:
errors_image = dict()
for base_path in errors.keys():
    folder = os.path.dirname(base_path)
    file = os.path.basename(base_path)
    output_path = get_output_path(folder, file)
    if not os.path.exists(output_path):
        try:
            print(file)
            pdf_images = get_pdf_images(base_path)
            text = [pytesseract.image_to_string(img, lang='"spa+eng') for img in pdf_images]
            process_pdf(text, base_path, file)
        except Exception as e:
            errors_image[folder] = str(e)

BAJAJ PLATINA 100 MANUAL TALLER 2.pdf
BAJAJ PULSAR 180 MANUAL TALLER.pdf
BAJAJ PULSAR NS200 - AS200 MANUAL USUARIO.pdf
BMW 650 DIAGRAMA ELECTRICO.jpg
BMW F800GS WIRING DIAGRAM 2010.jpg
BMW F800GS WIRING DIAGRAM.JPG
HARLEY DAVIDSON SPORTER (59-85 V-TWINS 86-87) GLIDES &ELECTRA GLIDES (59-84) EVOLUTION V-TWINS (84-86) MANUAL TALLER.pdf


KeyboardInterrupt: 